In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:
spark.sql("SHOW TABLES IN workspace.information_schema").show(truncate=False)
spark.sql("SHOW VOLUMES IN workspace.information_schema").show(truncate=False)

+------------------+-------------------------------+-----------+
|database          |tableName                      |isTemporary|
+------------------+-------------------------------+-----------+
|information_schema|catalog_privileges             |false      |
|information_schema|catalog_tags                   |false      |
|information_schema|catalogs                       |false      |
|information_schema|check_constraints              |false      |
|information_schema|column_masks                   |false      |
|information_schema|column_tags                    |false      |
|information_schema|columns                        |false      |
|information_schema|constraint_column_usage        |false      |
|information_schema|constraint_table_usage         |false      |
|information_schema|information_schema_catalog_name|false      |
|information_schema|key_column_usage               |false      |
|information_schema|parameters                     |false      |
|information_schema|refer

In [0]:

scored = spark.read.csv(
    "/Volumes/workspace/default/lending_club/exports/lc_scored.csv",
    header=True, inferSchema=True
)
scored.createOrReplaceTempView("scored")

In [0]:
scored.printSchema()

root
 |-- id: integer (nullable = true)
 |-- pd_prediction: double (nullable = true)
 |-- EAD: double (nullable = true)
 |-- EL: double (nullable = true)
 |-- interest_income: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- is_bad: integer (nullable = true)



In [0]:
print(scored.count(), "rows")
scored.show(5)

225639 rows
+---------+-------------------+-------+------------------+---------------+-----+------+
|       id|      pd_prediction|    EAD|                EL|interest_income|grade|is_bad|
+---------+-------------------+-------+------------------+---------------+-----+------+
|144528743| 0.3955882429619537|10800.0|3803.6758972574958|         2413.8|    D|     0|
|144268219| 0.1831536121580314| 3000.0|  489.184982712886|          507.3|    C|     0|
|144038203| 0.3600067949190461|24000.0| 7692.337188394241|         6544.8|    E|     0|
|143904190|0.19646316859967228|32000.0| 5597.157088137223|         4339.2|    C|     0|
|143652255|0.10922924563505387| 5000.0| 486.2339869444423|          516.5|    B|     0|
+---------+-------------------+-------+------------------+---------------+-----+------+
only showing top 5 rows


In [0]:
kpi_df = spark.sql("""
    SELECT
        COUNT(*)                       AS n_loans,
        SUM(EAD)                       AS total_ead,
        SUM(EL)                        AS total_el,
        SUM(EL) / SUM(EAD)             AS el_rate,
        (SUM(EL) / SUM(EAD)) * 10000   AS el_rate_bps
    FROM scored
""")
kpi_df.show()
kpi_df.toPandas().to_csv("/Volumes/workspace/default/lending_club/exports/portfolio_kpi.csv", index=False)

+-------+-----------+-------------------+-------------------+------------------+
|n_loans|  total_ead|           total_el|            el_rate|       el_rate_bps|
+-------+-----------+-------------------+-------------------+------------------+
| 225639|3.2595794E9|5.981144327722564E8|0.18349435904897926|1834.9435904897925|
+-------+-----------+-------------------+-------------------+------------------+



In [0]:
grade_df = spark.sql("""
    SELECT
        grade,
        COUNT(*)                       AS n_loans,
        SUM(EAD)                       AS total_ead,
        SUM(EL)                        AS total_el,
        SUM(EL) / SUM(EAD)             AS el_rate,
        (SUM(EL) / SUM(EAD)) * 10000   AS el_rate_bps
    FROM scored
    GROUP BY grade
    ORDER BY grade
""")
grade_df.show()
grade_df.toPandas().to_csv("/Volumes/workspace/default/lending_club/exports/grade_el_summary.csv", index=False)

+-----+-------+-------------+--------------------+-------------------+------------------+
|grade|n_loans|    total_ead|            total_el|            el_rate|       el_rate_bps|
+-----+-------+-------------+--------------------+-------------------+------------------+
|    A|  39710|  5.2479615E8| 2.815108460998181E7|0.05364194194256534| 536.4194194256534|
|    B|  62234| 8.31150525E8| 9.779024101151834E7|0.11765647505488654|1176.5647505488653|
|    C|  69674|1.021431175E9|2.0271397587756538E8|0.19846072925820518|1984.6072925820517|
|    D|  34196| 5.30745225E8|1.3949039489179146E8| 0.2628198772618095| 2628.198772618095|
|    E|  13246| 2.21541275E8| 7.500613233574148E7|0.33856504769028467|3385.6504769028465|
|    F|   4309|    8.25824E7|3.4005082054986276E7| 0.4117715403643667| 4117.715403643667|
|    G|   2270|   4.733265E7|  2.09575219906696E7| 0.4427709412143542| 4427.709412143542|
+-----+-------+-------------+--------------------+-------------------+------------------+



In [0]:
calib_df = spark.sql("""
    SELECT
        CONCAT(CAST(FLOOR(pd_prediction*10)*10 AS INT), '-',
               CAST(FLOOR(pd_prediction*10)*10 + 10 AS INT), '%') AS pd_band,
        FLOOR(pd_prediction*10) / 10           AS pd_band_floor,
        COUNT(*)                                AS n_loans,
        AVG(pd_prediction)                      AS avg_predicted_pd,
        AVG(is_bad)                             AS actual_default_rate,
        AVG(is_bad) - AVG(pd_prediction)        AS calibration_gap
    FROM scored
    GROUP BY 1, 2
    ORDER BY pd_band_floor
""")
calib_df.show()
calib_df.toPandas().to_csv("/Volumes/workspace/default/lending_club/exports/pd_calibration.csv", index=False)

+-------+-------------+-------+-------------------+-------------------+--------------------+
|pd_band|pd_band_floor|n_loans|   avg_predicted_pd|actual_default_rate|     calibration_gap|
+-------+-------------+-------+-------------------+-------------------+--------------------+
|  0-10%|          0.0|  51016|0.06703616851046176|0.07135016465422613|0.004313996143764362|
| 10-20%|          0.1|  81123|0.14850956722101868|0.17310750342073147| 0.02459793619971279|
| 20-30%|          0.2|  57191|0.24477094091802618| 0.2781206833242993|0.033349742406273114|
| 30-40%|          0.3|  23399|0.34203992339482236| 0.3595880165819052|0.017548093187082836|
| 40-50%|          0.4|   9361| 0.4420359265678778| 0.4407648755474842|-0.00127105102039...|
| 50-60%|          0.5|   3141| 0.5387992405434389| 0.5288124801018784|-0.00998676044156...|
| 60-70%|          0.6|    408| 0.6187036437395467| 0.6200980392156863|0.001394395476139...|
+-------+-------------+-------+-------------------+-------------------

In [0]:
decision_df = spark.sql("""
    SELECT
        CASE WHEN pd_prediction <= 0.12 THEN 'Approve' ELSE 'Decline' END AS decision,
        COUNT(*)                                       AS n_loans,
        SUM(interest_income)                           AS total_interest_income,
        SUM(EL)                                        AS total_el,
        SUM(interest_income) - SUM(EL)                 AS net_contribution
    FROM scored
    GROUP BY 1
    ORDER BY decision
""")
decision_df.show()
decision_df.toPandas().to_csv("/Volumes/workspace/default/lending_club/exports/approval_decision.csv", index=False)

+--------+-------+---------------------+-------------------+--------------------+
|decision|n_loans|total_interest_income|           total_el|    net_contribution|
+--------+-------+---------------------+-------------------+--------------------+
| Approve|  67447|  7.517110551250002E7|6.040650340458864E7|1.4764602107911378E7|
| Decline| 158192|   3.91610764774999E8|5.377079293676673E8|-1.460971645926683E8|
+--------+-------+---------------------+-------------------+--------------------+

